# 00 เริ่มต้นและรู้จักข้อมูลตาราง
ทำได้ก่อนมี dataset จากอาจารย์ ตัวเลขใน notebook นี้สร้างขึ้นเพื่อฝึกเท่านั้น ไม่ใช่ข้อมูลหรือผลโครงงาน

เป้าหมาย: รัน cell ได้ แยก input/target ได้ และอธิบายค่าเฉลี่ยโดยไม่ต้องท่องคำสั่ง อ่าน docs/START_HERE_TH ก่อนเริ่ม


## เจ้าของงานและการส่งต่อสำหรับทีม 5 คน
ทุกคนทำแบบฝึกนี้และตกลง M1–M5 ก่อนเริ่มงาน ดู Workflow สำหรับสมาชิก 5 คนใน docs/WORKFLOW_5_MEMBERS_TH.md

M1–M5 เป็นรหัสบทบาทเสนอ ให้กลุ่มใส่ชื่อจริงและตกลงผู้ตรวจ ไม่ใช่หลักฐานว่างานทำเสร็จแล้ว ทุกคนบันทึก contribution/prompts และเขียนรายงานส่วนของตน


## เตรียมงานใน Colab
บันทึกสำเนา notebook ใน Drive ก่อน รัน cell นี้เพื่อเตรียม repository ใน runtime เมื่อมีโฟลเดอร์เดิมจะใช้ต่อโดยไม่เขียนทับ สำรองไฟล์ใน Files แยกจาก notebook ก่อนปิด runtime

ใช้ CPU runtime ได้ ไม่ต้องติดตั้งโปรแกรมในเครื่อง


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/Bicycle_Rental_Prediction')]
ROOT = next((p for p in candidates if (p / 'config/project.json').is_file()), None)
if ROOT is None:
    if not Path('/content').is_dir():
        raise RuntimeError('เปิด notebook นี้บน Google Colab และเชื่อม runtime ก่อน')
    ROOT = Path('/content/Bicycle_Rental_Prediction')
    if ROOT.exists():
        raise FileExistsError('พบโฟลเดอร์เดิมที่ยังอ่าน config ไม่ได้ ตรวจและสำรองงานก่อนแก้')
    subprocess.run(['git', 'clone',
                    'https://github.com/Supot77/Bicycle_Rental_Prediction.git',
                    str(ROOT)], check=True)

# Keep Colab's existing scientific stack; install only packages that are missing.
packages = {'pandas': 'pandas', 'sklearn': 'scikit-learn',
            'matplotlib': 'matplotlib', 'joblib': 'joblib'}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from starter_support import load_config, configuration_problems, resolve_dataset_path
config = load_config(ROOT)
problems = configuration_problems(config)
if problems:
    raise ValueError('\n'.join(problems))
print('พร้อมทำงานบน Colab: ใช้โฟลเดอร์ repository เดิมถ้ามี โดยไม่ pull/reset ทับงาน')
print('ค่า null ใน config เป็นสถานะรอยืนยัน ไม่ใช่ข้อมูลที่ต้องเดา')


## แถว คอลัมน์ และ target
หนึ่งแถวควรมีความหมายแน่นอน เช่น หนึ่งช่วงเวลา ส่วนคอลัมน์เป็นคุณสมบัติของแถวนั้น Target คือคำตอบที่ต้องการทำนาย ส่วน feature คือสิ่งที่จะรู้ตอนทำนาย ห้ามนำคำตอบเข้ามาเป็น input

ตัวอย่างข้างล่างใช้เพียงรายการยอดสมมติ เพื่อเรียนรู้การอ่านตัวเลข ยังไม่มี feature หรือการฝึกโมเดล


In [ ]:
# ข้อมูลสมมติเพื่อฝึก Python พื้นฐานเท่านั้น
practice_rentals = [10, 20, 30, 20]
print('จำนวนรายการ:', len(practice_rentals))
print('ผลรวม:', sum(practice_rentals))
print('ค่าเฉลี่ย:', sum(practice_rentals) / len(practice_rentals))


## ลองด้วยตนเอง
1. คำนวณค่าเฉลี่ยด้วยมือก่อนรัน และเขียนว่าผลตรงกันหรือไม่
2. เปลี่ยนตัวเลขหนึ่งค่าให้สูงขึ้น แล้วอธิบายผลต่อค่าเฉลี่ย
3. ถ้ามีค่าหาย ไม่ควรเปลี่ยนเป็นศูนย์ทันทีเพราะอะไร
4. ค่าเฉลี่ยจาก train จะใช้เป็น baseline อย่างไร และทำไมไม่ใช้ค่าเฉลี่ยจาก test มาทำนายทุกแถว

เพิ่ม code cell เพื่อทดลอง แล้วเขียนคำตอบด้วยภาษาตนเองด้านล่าง


## พื้นที่บันทึกการเรียนรู้
เขียนสิ่งที่ลอง ผลที่เห็น และสิ่งที่ยังสงสัย ไม่จำเป็นต้องตอบเหมือนเพื่อน ขอให้ตรวจจากการทดลองได้


## พร้อมไป notebook 01 เมื่อ
เปิดและรัน notebook ได้ อธิบาย feature/target/baseline ได้ และรู้ว่ากลุ่มต้องรอ dataset จริงก่อนตัดสินใจเรื่องคอลัมน์

แบ่งงานเตรียมตัวใน evidence/task_board.csv ได้ทันที


## ก่อนส่งต่อ
เก็บข้อสังเกตและหลักฐานที่เผยแพร่ได้ไว้ใน evidence/results ให้เพื่อนอีกคนตรวจและลองทำซ้ำ จากนั้นล้าง outputs และ execution counts ก่อน commit notebook อย่าใส่ผลสมมติเป็นผลทดลองจริง
